# P5-4 · GĐ 4 · Latent diffusion và Stable Diffusion

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers

## Lý thuyết

In [ ]:
# Cần GPU và mạng: tải Stable Diffusion 1.5 (khoảng 2 GB ở fp16), chạy trên Colab T4
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
dev = "cuda"
pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                               torch_dtype=torch.float16).to(dev)
tok, te, unet, vae = pipe.tokenizer, pipe.text_encoder, pipe.unet, pipe.vae
sched = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

@torch.no_grad()
def encode(texts):
    ids = tok(texts, padding="max_length", max_length=tok.model_max_length,
              truncation=True, return_tensors="pt").input_ids.to(dev)
    return te(ids)[0]                                            # (B, 77, 768)

@torch.no_grad()
def generate(prompt, negative="", steps=30, w=7.5, seed=42, h=512, wd=512):
    emb = encode([negative, prompt])                             # 1. nhánh không điều kiện + có điều kiện
    g = torch.Generator(dev).manual_seed(seed)                   # 2. nhiễu ban đầu từ seed
    z = torch.randn(1, 4, h // 8, wd // 8, generator=g, device=dev, dtype=torch.float16)
    sched.set_timesteps(steps)                                   # 3. chọn các bước (999, 966, 932, ...)
    z = z * sched.init_noise_sigma
    for t in sched.timesteps:                                    # 4. lặp
        inp = sched.scale_model_input(torch.cat([z, z]), t)
        eps_u, eps_c = unet(inp, t, encoder_hidden_states=emb).sample.chunk(2)
        eps = eps_u + w * (eps_c - eps_u)                        #    CFG
        z = sched.step(eps, t, z).prev_sample                    #    scheduler cập nhật latent
    img = vae.decode(z / vae.config.scaling_factor).sample       # 5. giải mã
    img = ((img.clamp(-1, 1) + 1) / 2)[0].permute(1, 2, 0).float().cpu().numpy()
    return (img * 255).round().astype("uint8")                   # (512, 512, 3)

In [ ]:
import torch
from diffusers import UNet2DConditionModel, AutoencoderKL, DPMSolverMultistepScheduler
torch.manual_seed(0)
unet = UNet2DConditionModel(sample_size=16, block_out_channels=(32, 64), layers_per_block=1, cross_attention_dim=32,
                            down_block_types=("CrossAttnDownBlock2D", "DownBlock2D"),
                            up_block_types=("UpBlock2D", "CrossAttnUpBlock2D"), attention_head_dim=8)
vae = AutoencoderKL(block_out_channels=(32, 64), down_block_types=("DownEncoderBlock2D",) * 2,
                    up_block_types=("UpDecoderBlock2D",) * 2, latent_channels=4, norm_num_groups=16)
sched = DPMSolverMultistepScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear")
emb = torch.randn(2, 77, 32)                                   # [nhánh không điều kiện, nhánh có điều kiện]

z = torch.randn(1, 4, 16, 16) * sched.init_noise_sigma
sched.set_timesteps(30)
print("các bước:", sched.timesteps[:4].tolist(), "...")
with torch.no_grad():
    for t in sched.timesteps:
        inp = sched.scale_model_input(torch.cat([z, z]), t)
        eps_u, eps_c = unet(inp, t, encoder_hidden_states=emb).sample.chunk(2)
        z = sched.step(eps_u + 7.5 * (eps_c - eps_u), t, z).prev_sample
    img = vae.decode(z / 0.18215).sample
print("latent", tuple(z.shape), "→ ảnh", tuple(img.shape))     # VAE nhỏ này giảm 2 lần mỗi cạnh; VAE của SD giảm 8 lần

các bước: [999, 966, 932, 899] ...
latent (1, 4, 16, 16) → ảnh (1, 3, 32, 32)


## Bài tập 4: Đọc pipeline {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng: tải Stable Diffusion 1.5, chạy trên Colab
import torch
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16)
count = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"U-Net {count(pipe.unet):.0f}M | text encoder {count(pipe.text_encoder):.0f}M | VAE {count(pipe.vae):.0f}M")
print("scaling factor:", pipe.vae.config.scaling_factor)
print("scheduler:", type(pipe.scheduler).__name__, "| max tokens:", pipe.tokenizer.model_max_length)
print("cross-attention dim:", pipe.unet.config.cross_attention_dim)

## Bài tập 5: Nén ảnh vào không gian latent {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
X = torch.tensor(load_digits().images / 16, dtype=torch.float32).unsqueeze(1)
X_tr, X_te = X[:1500], X[1500:]
torch.manual_seed(0)
enc = nn.Sequential(nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.SiLU(), nn.Conv2d(16, 4, 3, stride=2, padding=1))   # → 4×2×2
dec = nn.Sequential(nn.ConvTranspose2d(4, 16, 2, stride=2), nn.SiLU(), nn.ConvTranspose2d(16, 1, 2, stride=2), nn.Sigmoid())
# TODO: train 1500 bước (batch 64, Adam 3e-3, loss MSE); he_so_nen; mse_test

In [ ]:
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=3e-3)
for step in range(1500):
    xb = X_tr[torch.randint(0, 1500, (64,))]
    loss = ((dec(enc(xb)) - xb) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
z = enc(X[:1])
he_so_nen = X[0].numel() // z[0].numel()
with torch.no_grad():
    rec = dec(enc(X_te))
    mse_test = ((rec - X_te) ** 2).mean().item()
fig, axes = plt.subplots(2, 6, figsize=(8, 2.8))
for j in range(6):
    axes[0, j].imshow(X_te[j, 0], cmap="gray_r"); axes[1, j].imshow(rec[j, 0], cmap="gray_r")
    axes[0, j].axis("off"); axes[1, j].axis("off")
print(f"latent {tuple(z.shape)}, nén {he_so_nen} lần, MSE test {mse_test:.4f}")

latent (1, 4, 2, 2), nén 4 lần, MSE test 0.0134


## Bài tập 6: Đếm số lần chạy U-Net trong vòng lặp sinh ảnh {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
from diffusers import UNet2DConditionModel, DPMSolverMultistepScheduler
torch.manual_seed(0)
unet = UNet2DConditionModel(sample_size=8, in_channels=4, out_channels=4, layers_per_block=1,
                            block_out_channels=(32, 64), down_block_types=("CrossAttnDownBlock2D", "DownBlock2D"),
                            up_block_types=("UpBlock2D", "CrossAttnUpBlock2D"), cross_attention_dim=32, attention_head_dim=8).eval()
sched = DPMSolverMultistepScheduler(num_train_timesteps=1000)
text_emb = torch.randn(1, 7, 32)          # thay cho đầu ra của text encoder
null_emb = torch.zeros(1, 7, 32)          # embedding của prompt rỗng
dem = {"goi": 0}
unet.register_forward_hook(lambda m, i, o: dem.__setitem__("goi", dem["goi"] + 1))
# TODO: set_timesteps(20); latent ngẫu nhiên × init_noise_sigma; vòng lặp có CFG (w = 7.5) ghép batch 2

In [ ]:
sched.set_timesteps(20)
lat = torch.randn(1, 4, 8, 8) * sched.init_noise_sigma
with torch.no_grad():
    for t in sched.timesteps:
        inp = sched.scale_model_input(torch.cat([lat, lat]), t)
        e = unet(inp, t, encoder_hidden_states=torch.cat([null_emb, text_emb])).sample
        e_rong, e_chu = e.chunk(2)
        e = e_rong + 7.5 * (e_chu - e_rong)
        lat = sched.step(e, t, lat).prev_sample
print("số bước:", len(sched.timesteps), "| lần gọi U-Net:", dem["goi"], "| latent:", tuple(lat.shape))
print("nếu chạy riêng có chữ và không chữ: ", 2 * len(sched.timesteps), "lần gọi")

số bước: 20 | lần gọi U-Net: 20 | latent: (1, 4, 8, 8)
nếu chạy riêng có chữ và không chữ:  40 lần gọi


## Bài tập 7: img2img: strength quyết định bắt đầu từ đâu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
from diffusers import DDIMScheduler
sched = DDIMScheduler(num_train_timesteps=1000, beta_schedule="scaled_linear", beta_start=0.00085, beta_end=0.012)
sched.set_timesteps(50)
# TODO: với mỗi strength: so_buoc_chay = int(50 * strength); bắt đầu ở chỉ số 50 - so_buoc_chay; t_bat_dau; tin_hieu

In [ ]:
kq = {}
for s in (0.3, 0.6, 0.9):
    n = int(50 * s)
    t0 = int(sched.timesteps[50 - n])
    kq[s] = (n, t0, sched.alphas_cumprod[t0].sqrt().item())
    print(f"strength {s}: chạy {n} bước, bắt đầu ở t = {t0}, √ᾱ = {kq[s][2]:.3f}")

strength 0.3: chạy 15 bước, bắt đầu ở t = 280, √ᾱ = 0.790
strength 0.6: chạy 30 bước, bắt đầu ở t = 580, √ᾱ = 0.425
strength 0.9: chạy 45 bước, bắt đầu ở t = 880, √ᾱ = 0.132
